<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Pinball Loss, Explained · How a Model Learns a Low, an Expected and a High
### A small weather and energy example (optional, about 20 minutes)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab6_Pinball_Loss_Explained.ipynb)

Your range model gives **three numbers for each of the next 24 hours**: a low, an expected and a high. How does a
model "know" what a 10th percentile is? This notebook answers that with one idea, the **pinball loss**, and a small
example: summer afternoons at Bradley Airport and Connecticut's electricity load.

| Part | What you see |
|---|---|
| **1. The formula** | Pinball loss in one line, and why it's lopsided on purpose |
| **2. The picture** | The "hockey stick" shape for the low, expected and high |
| **3. The aha** | The forecast with the lowest pinball loss IS the percentile |
| **4. Add temperature** | Three lines through a scatterplot, one per percentile |
| **5. 24 hours, 3 numbers each** | One model, every hour is just a row |
| **6. Back to the leaderboard** | The CRPS you're scored on = your three pinball losses added up |

Just run each cell. Nothing here is graded.

# Setup
The same Lab 5 data: hourly weather measured at Bradley and Connecticut's load since 2023.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor

DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])

# Connecticut time, and a few calendar columns
local = df.index.tz_convert("America/New_York")
df["year"] = local.year
df["month"] = local.month
df["hour"] = local.hour
df["weekday"] = local.dayofweek < 5
holidays = USFederalHolidayCalendar().holidays(start="2023-01-01", end="2026-12-31")
df["holiday"] = local.tz_localize(None).normalize().isin(holidays)

# Our small example: summer workdays (June to August, Monday to Friday, no holidays), 2023 to 2025
summer = df[(df["month"] >= 6) & (df["month"] <= 8) & df["weekday"] & ~df["holiday"] & (df["year"] <= 2025)]

# Even smaller: just 5 PM, when air conditioners are running hard
five_pm = summer[summer["hour"] == 17]
print(len(five_pm), "summer workday afternoons at 5 PM, 2023 to 2025")
print("Load at 5 PM ranged from", round(five_pm["load_mw"].min()), "to", round(five_pm["load_mw"].max()), "MW")

# Part 1 · The formula
Say a forecast misses by **miss = actual − forecast** (in MW). Pick a percentile **q** (0.1 for the low, 0.5 for the
expected, 0.9 for the high). The pinball loss for one hour is:

| If the actual came in... | You forecast | Pinball loss |
|---|---|---|
| **above** your forecast (miss > 0) | too low | **q × miss** |
| **below** your forecast (miss < 0) | too high | **(1 − q) × (−miss)** |

That's it. The only trick is that the two sides cost **different amounts**:

- **High (q = 0.9):** too low costs **0.9** per MW, too high costs only **0.1**. Being under is 9 times as painful, so
  the model pushes its high **up** until only about 1 hour in 10 lands above it.
- **Low (q = 0.1):** the opposite. Too high is 9 times as painful, so the low gets pushed **down**.
- **Expected (q = 0.5):** both sides cost 0.5. That's just half the absolute error (Lab 5's MAE), and it lands on the
  **median**.

Try it: one hour, the actual load is 3,000 MW.

In [ ]:
def pinball(actual, forecast, q):
    """Pinball loss for one percentile q. Works on single numbers or whole columns."""
    miss = np.asarray(actual, dtype=float) - np.asarray(forecast, dtype=float)
    too_low = q * miss                 # used when the actual came in above the forecast
    too_high = (1 - q) * (-miss)       # used when the actual came in below the forecast
    return np.where(miss > 0, too_low, too_high)


actual = 3000
rows = []
for forecast in [2800, 3200]:
    for q in [0.1, 0.5, 0.9]:
        rows.append({"forecast (MW)": forecast,
                     "q": q,
                     "miss = actual - forecast": actual - forecast,
                     "pinball loss": float(pinball(actual, forecast, q))})
pd.DataFrame(rows)

**Read it:** both forecasts missed by 200 MW, but the pinball loss depends on **which way** and **which q**.
- Forecast 2,800 (too low): the **high** (q = 0.9) pays 0.9 × 200 = **180**. The **low** (q = 0.1) pays only 20: a low
  is *supposed* to sit under the actual most of the time.
- Forecast 3,200 (too high): flipped. The low pays 180, the high pays only 20.
- The expected (q = 0.5) pays 100 either way.

# Part 2 · The picture
Plot the loss against the miss and you get a lopsided V: the "pinball" (some people say "hockey stick"). The steep
side is the mistake that percentile is told to avoid.

In [ ]:
misses = np.linspace(-500, 500, 201)        # actual - forecast, in MW
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
names = ["low (q = 0.1)", "expected (q = 0.5)", "high (q = 0.9)"]
colors = ["tab:blue", "tab:gray", "tab:orange"]
quantiles = [0.1, 0.5, 0.9]
for i in range(3):
    loss = pinball(misses, 0, quantiles[i])
    axes[i].plot(misses, loss, color=colors[i], linewidth=3)
    axes[i].axvline(0, color="black", linewidth=0.8)
    axes[i].set_title(names[i])
    axes[i].set_xlabel("miss = actual - forecast (MW)")
    axes[i].text(-480, 420, "forecast\ntoo high", fontsize=9)
    axes[i].text(300, 420, "forecast\ntoo low", fontsize=9)
axes[0].set_ylabel("pinball loss")
plt.suptitle("Pinball loss: each percentile is punished more on one side")
plt.tight_layout()
plt.show()

- **Low:** steep on the **left** (forecast too high hurts), gentle on the right.
- **Expected:** a symmetric V, the absolute error cut in half.
- **High:** steep on the **right** (forecast too low hurts), gentle on the left.

# Part 3 · The aha: the lowest pinball loss lands on the percentile
Take every hot summer afternoon (5 PM, **85 to 92 °F**). If you had to forecast **one number** for all of them, which
number gives the lowest **average** pinball loss? Try every number from 3,000 to 6,500 MW and see.

In [ ]:
hot = five_pm[(five_pm["temp_f"] >= 85) & (five_pm["temp_f"] <= 92)]
loads = hot["load_mw"].values
print(len(loads), "hot afternoons")

candidates = np.arange(3000, 6501, 5)        # every forecast we try, in MW
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))

# Left: what really happened on those afternoons, with the 10th, 50th and 90th percentile
axes[0].hist(loads, bins=25, color="lightgray", edgecolor="black")
for i in range(3):
    axes[0].axvline(np.percentile(loads, 100 * quantiles[i]), color=colors[i], linewidth=3, label=names[i] + " percentile")
axes[0].set_xlabel("load at 5 PM (MW)")
axes[0].set_ylabel("number of afternoons")
axes[0].set_title("What really happened")
axes[0].legend(fontsize=8)

# Right: the average pinball loss of every candidate forecast, for each q
best = []
for i in range(3):
    average_loss = []
    for c in candidates:
        average_loss.append(np.mean(pinball(loads, c, quantiles[i])))
    winner = candidates[int(np.argmin(average_loss))]
    best.append(winner)
    axes[1].plot(candidates, average_loss, color=colors[i], linewidth=2.5, label=names[i])
    axes[1].axvline(winner, color=colors[i], linestyle=":", linewidth=2)
axes[1].set_xlabel("one forecast for every hot afternoon (MW)")
axes[1].set_ylabel("average pinball loss")
axes[1].set_title("The bottom of each curve = the best forecast for that q")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for i in range(3):
    print(names[i].ljust(20), "lowest pinball loss at", best[i], "MW | percentile of the data:", round(np.percentile(loads, 100 * quantiles[i])), "MW")

**The bottom of each curve sits on the percentile** (the dotted lines on the right match the solid lines on the left,
within the 5 MW steps we tried). That's the whole trick, and it's why the pinball loss is the standard score for this
job: **the forecast that minimizes the pinball loss for q IS the q-th percentile.** So a model trained to minimize it
for q = 0.9 learns to put about 9 hours in 10 below its high. (Koenker and Bassett, 1978, "Regression Quantiles".)

Why? Nudge the high up by 1 MW. The hours above it each save 0.9; the hours below it each cost 0.1 more. Keep going
until the savings and costs balance: 0.9 × (share above) = 0.1 × (share below), which happens when 10% are above.

# Part 4 · Add temperature: three lines through a scatterplot
One number for every afternoon ignores the weather. Let the forecast depend on temperature: a **linear quantile
regression** is a straight line chosen to minimize the pinball loss. Fit it three times (q = 0.1, 0.5, 0.9).

In [ ]:
X = five_pm[["temp_f"]]
y = five_pm["load_mw"]
grid_temps = pd.DataFrame({"temp_f": np.linspace(X["temp_f"].min(), X["temp_f"].max(), 50)})

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X["temp_f"], y, s=14, color="black", alpha=0.5, label="actual (one dot per afternoon)")
lines = {}
for i in range(3):
    model = QuantileRegressor(quantile=quantiles[i], alpha=0, solver="highs")     # alpha=0: no penalty, just pinball loss
    model.fit(X, y)
    lines[quantiles[i]] = model
    ax.plot(grid_temps["temp_f"], model.predict(grid_temps), color=colors[i], linewidth=3, label=names[i])
ax.set_xlabel("temperature at 5 PM (°F)")
ax.set_ylabel("load at 5 PM (MW)")
ax.set_title("Summer workdays at 5 PM: one line per percentile")
ax.legend()
plt.show()

for i in range(3):
    share_below = np.mean(y.values <= lines[quantiles[i]].predict(X))
    print(names[i].ljust(20), "share of afternoons below this line:", str(round(100 * share_below)) + "%")

**Read it:** the three lines climb with temperature (air conditioning), and about 10%, 50% and 90% of the dots sit
below the low, expected and high line. Between the low and the high: about **80%** of afternoons, your 80% range.

# Part 5 · 24 hours, 3 numbers each
Here's the part that can feel like magic: forecasting the next 24 hours does **not** mean 24 separate models. Every
hour is just a **row** of inputs (the hour of the day, the temperature forecast, ...). The same model reads each row
and gives its 3 numbers. 24 rows × 3 numbers = 72 numbers, one model per percentile.

Below, gradient boosting with `loss="quantile"` learns from **every hour** of those summer workdays, with two inputs:
the hour of the day and the temperature. Then it forecasts one hot day in 2026 it never saw. (In production your model
reads the **NWS forecast** for the temperature; here we use the measured temperature to keep it simple.)

In [ ]:
inputs = ["hour", "temp_f"]
models = {}
for q in quantiles:
    models[q] = GradientBoostingRegressor(loss="quantile", alpha=q, n_estimators=200, max_depth=3, random_state=42)
    models[q].fit(summer[inputs], summer["load_mw"])

# One hot summer workday in 2026: the 24 rows the model reads
summer_2026 = df[(df["year"] == 2026) & (df["month"] >= 6) & (df["month"] <= 8) & df["weekday"] & ~df["holiday"]]
hottest_day = summer_2026["temp_f"].idxmax().tz_convert("America/New_York").date()
day = summer_2026[summer_2026.index.tz_convert("America/New_York").date == hottest_day]
print("The 24 rows for", hottest_day, "(first 5 shown):")
print(day[inputs].head())

forecast = pd.DataFrame(index=day.index)
forecast["low"] = models[0.1].predict(day[inputs])
forecast["expected"] = models[0.5].predict(day[inputs])
forecast["high"] = models[0.9].predict(day[inputs])
forecast["actual"] = day["load_mw"]
hours = day["hour"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(hours, day["temp_f"], color="tab:red", linewidth=2.5)
axes[0].set_xlabel("hour of the day (Connecticut time)")
axes[0].set_ylabel("temperature (°F)")
axes[0].set_title("The input: temperature, hour by hour")
axes[1].fill_between(hours, forecast["low"], forecast["high"], color="tab:blue", alpha=0.25, label="low to high (80% range)")
axes[1].plot(hours, forecast["expected"], color="tab:blue", linewidth=2.5, label="expected")
axes[1].plot(hours, forecast["actual"], color="black", linewidth=2.5, label="actual")
axes[1].set_xlabel("hour of the day (Connecticut time)")
axes[1].set_ylabel("load (MW)")
axes[1].set_title("The output: 3 numbers for each of 24 hours")
axes[1].legend(fontsize=8)
plt.suptitle(str(hottest_day) + ", a day the models never saw")
plt.tight_layout()
plt.show()

inside = (forecast["actual"] >= forecast["low"]) & (forecast["actual"] <= forecast["high"])
print("Hours inside the range:", int(inside.sum()), "of", len(forecast))

**What about the quantile forest?** The final project's model (Lab 6B) gets there a different way: each leaf of each
tree keeps **every real load** of the training hours that landed in it, and the forest reads off the 10th, 50th and
90th percentile of those loads. It never computes a pinball loss while it trains, but Part 3 shows why it aims at the
same target: the percentile of what really happened is exactly the forecast with the lowest pinball loss. (Meinshausen,
2006, "Quantile Regression Forests".) That's also why one quantile forest can give all three numbers at once.

# Part 6 · Back to the leaderboard
The leaderboard's CRPS for one hour is:

**CRPS = ( 0.5 × |actual − expected| + 0.1 × width + 1 × outside ) ÷ 1.5**

It's the same thing as **your three pinball losses added up**, divided by 1.5:

**CRPS = ( pinball(low, q = 0.1) + pinball(expected, q = 0.5) + pinball(high, q = 0.9) ) ÷ 1.5**

Check it on the four forecasters from Lab 6B (all expected 3,000 MW, actual 3,100 MW):

In [ ]:
def leaderboard_crps(actual, low, expected, high):
    """The leaderboard's formula (Lab 6B Part 1)."""
    width = high - low
    outside = max(low - actual, 0) + max(actual - high, 0)
    return (0.5 * abs(actual - expected) + 0.1 * width + 1 * outside) / 1.5


def crps_from_pinball(actual, low, expected, high):
    """The same score, built from three pinball losses."""
    total = pinball(actual, low, 0.1) + pinball(actual, expected, 0.5) + pinball(actual, high, 0.9)
    return float(total) / 1.5


forecasters = pd.DataFrame({"forecaster": ["no range (Lab 5)", "too narrow", "honest", "too wide"],
                            "low": [3000, 2980, 2850, 2000],
                            "expected": [3000, 3000, 3000, 3000],
                            "high": [3000, 3020, 3200, 4000]})
leaderboard_scores = []
pinball_scores = []
for i in range(len(forecasters)):
    row = forecasters.iloc[i]
    leaderboard_scores.append(round(leaderboard_crps(3100, row["low"], row["expected"], row["high"]), 1))
    pinball_scores.append(round(crps_from_pinball(3100, row["low"], row["expected"], row["high"]), 1))
forecasters["leaderboard CRPS"] = leaderboard_scores
forecasters["from 3 pinball losses"] = pinball_scores
forecasters

**Same numbers.** Why it works for an hour where the actual lands **inside** the range:
- the low (q = 0.1) sits under the actual: 0.1 × (actual − low)
- the high (q = 0.9) sits over the actual: 0.1 × (high − actual)
- those two add up to **0.1 × width**, and the expected adds **0.5 × |actual − expected|**

When the actual lands **outside**, say above the high, the high's steep side kicks in (0.9 per MW instead of 0.1), and
that extra is the **1 × outside** term. So every time you tune for a lower CRPS, you're tuning your three numbers to
be better percentiles. (This sum of pinball losses is the weighted interval score of Bracher et al., 2021, used to
score the U.S. COVID-19 forecast hub.)

### The one-sentence version for your report
*Each of my three numbers is scored with a lopsided penalty (the pinball loss) that is smallest when it sits at its
percentile, so a low CRPS means my low, expected and high behave like a real 10th, 50th and 90th percentile.*

### Think about it (optional)
1. In Part 1, if you made q = 0.95 for the high, how much more would a too-low forecast cost than a too-high one? Would
   your range get wider or narrower?
2. In Part 3, change the temperature window to 70 to 78 °F. Do the three percentiles move closer together or farther
   apart? Why might that be?
3. In Part 5, which hours of the day have the widest range? Does that match what you see on the leaderboard?

---
**Sources:** Koenker and Bassett (1978), Regression Quantiles, *Econometrica*. Meinshausen (2006), Quantile
Regression Forests, *JMLR*. Bracher, Ray, Gneiting and Reich (2021), Evaluating epidemic forecasts in an interval
format, *PLOS Computational Biology*. Data: Iowa State Mesonet (Bradley weather) and ISO New England (load).